## Setup

In [1]:
# ── Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

# ── Dependency Guard ───────────────────────────────────────────────
# Ensures all required packages are installed in the active kernel.
# Safe to re-run: pip will skip already-installed packages.

import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical plotting
import fastf1                    # Formula 1 data access

print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

Python  : 3.14.2
NumPy   : 2.3.4
Seed    : 414
All required packages already installed ✓
fastf1  : 3.8.1
pandas  : 2.3.3


In [2]:
# Load and prepare 2020-2024 race-result data from FastF1/Ergast for EDA.

# ── YOUR TURN ──────────────────────────────────────────────────────────────
# Task: Implement majority-class and domain heuristic baselines on YOUR data.
# 
# Step 1: Load your Lab 1 data (copy your data loading code here)

# Configure local cache so repeated API reads are fast and reproducible.
cache_path = os.path.abspath("fastf1_cache")
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)

# Create Ergast interface and iterate all race weekends in each season.
erg = fastf1.ergast.Ergast()

years = [2020, 2021, 2022, 2023, 2024]
results_list = []

for year in years:
    schedule = fastf1.get_event_schedule(year)

    for _, event in schedule.iterrows():
        # Ignore testing sessions because they are not part of race prediction.
        if event['EventFormat'] == 'testing':
            continue

        rnd = int(event['RoundNumber'])

        try:
            res = erg.get_race_results(season=year, round=rnd)
            race = res.content[0]

            # Keep temporal/event metadata for later grouping and split logic.
            race['year'] = year
            race['round'] = rnd
            race['circuit'] = event['EventName']

            results_list.append(race)

        except Exception as e:
            print(f"Skipping {year} round {rnd}")

# Merge all race result frames into one modeling table.
df = pd.concat(results_list, ignore_index=True)

# Keep only columns used in this lab.
df = df[[
    'driverNumber',
    'givenName',
    'familyName',
    'constructorName',
    'grid',
    'position',
    'status',
    'points',
    'year',
    'round',
    'circuit'
]]

# Standardize column names for readability.
df.columns = [
    'DriverNumber',
    'FirstName',
    'LastName',
    'TeamName',
    'GridPosition',
    'Position',
    'Status',
    'Points',
    'Year',
    'Round',
    'Circuit'
]

# Build convenient identity and numeric fields.
df['FullName'] = df['FirstName'] + " " + df['LastName']
df['GridPosition'] = pd.to_numeric(df['GridPosition'], errors='coerce')
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')

# Feature engineering for baseline and analysis.
df['Top10'] = ((df['Position'] <= 10) & (df['Position'] >0)).astype(int)
df['BaselinePred'] = (df['GridPosition'] <= 10).astype(int)
df['PositionsGained'] = df['GridPosition'] - df['Position']
df['DNF'] = ~df['Status'].str.contains("Finished", na=False)

# Historical reliability and constructor strength proxies.
driver_reliability = 1 - df.groupby('FullName')['DNF'].mean()
df['DriverReliability'] = df['FullName'].map(driver_reliability)
constructor_points = df.groupby('TeamName')['Points'].mean()
df['ConstructorStrength'] = df['TeamName'].map(constructor_points)

# Remove unusable rows for grid/finish-based analyses.
df = df.dropna(subset=['GridPosition','Position'])

print("Dataset shape:", df.shape)


Request for URL https://api.jolpi.ca/ergast/f1/2023/8/results.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests_cache\session.py", line 297, in _resend
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests\models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2023/8/results.json
Request for URL https://api.jolpi.ca/ergast/f1/2023/9/results.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests_cache\session.py", line 297, in _resend
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\gotts\AppData\Local\

Dataset shape: (2139, 18)


MARTIN!!! Do this splits for train/test/validate

In [3]:
#We replace string values into int with get_dummies() 
df_trunq = df[['Top10','FullName','TeamName','GridPosition','Year','Circuit', 'Round']].copy()


print("Dataset shape:", df_trunq.shape)

df_trunq.head()

Dataset shape: (2139, 7)


,Top10,FullName,TeamName,GridPosition,Year,Circuit,Round
0,1,Valtteri Bottas,Mercedes,1,2020,Austrian Grand Prix,1
1,1,Charles Leclerc,Ferrari,7,2020,Austrian Grand Prix,1
2,1,Lando Norris,McLaren,3,2020,Austrian Grand Prix,1
3,1,Lewis Hamilton,Mercedes,5,2020,Austrian Grand Prix,1
4,1,Carlos Sainz,McLaren,8,2020,Austrian Grand Prix,1


In [4]:
#splitting by Year, choice was for recency. 2023 to train (its a past season). 2024 to evaluate (test)
df_train = df_trunq[df_trunq["Year"].isin([2020,2021,2022])].copy()
df_val = df_trunq[df_trunq["Year"]==2023].copy()
df_test = df_trunq[df_trunq["Year"]==2024].copy()
print(f"Training shape: {df_train.shape}\nValidation shape: {df_val.shape}\nTesting shape: {df_test.shape}")


Training shape: (1220, 7)
Validation shape: (440, 7)
Testing shape: (479, 7)


## Domain Heuristics

Firstly, a function to evaluate the baselines

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression #Binary prediction with LogisticRegression


def evaluate_baseline(name, df_eval): # La podriamos mover arriba
    print(f"\n{name}")
    #splits by year
    df_train = df_eval[df_eval["Year"].isin([2020,2021, 2022])].copy()
    df_val = df_eval[df_eval["Year"] == 2023].copy()
    df_test = df_eval[df_eval["Year"] == 2024].copy()
    print(f"Training shape: {df_train.shape}\nValidation shape: {df_val.shape}\nTesting shape: {df_test.shape}")

    #splits (X,y)
    y_train = df_train["Top10"]
    X_train = df_train.drop(["Top10",'Year'], axis=1)

    y_val = df_val["Top10"] 
    X_val = df_val.drop(["Top10",'Year'], axis=1)

    y_test = df_test["Top10"] 
    X_test = df_test.drop(["Top10",'Year'], axis=1)

    #scaler
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    #model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)


    #Results:
    print(classification_report(
        y_test, y_pred,
        target_names=['Not Top-10', 'Top-10'],
        zero_division=0,
    ))

    #print(f"Accuracy : {accuracy_score(y_val, y_pred):.3f}")
    #print(f"Precision: {precision_score(y_val, y_pred, zero_division=0):.3f}")
    #print(f"Recall   : {recall_score(y_val, y_pred, zero_division=0):.3f}")
    #print(f"F1       : {f1_score(y_val, y_pred, zero_division=0):.3f}")

### 1. "Everyone that ends on top 10, ALWAYS ends in the top 10"

In [6]:
# Baseline 1: predict Top10 if the driver starts in the first 10 grid slots.
#df_heuristic1 = df[['Top10','FullName','TeamName','GridPosition','Year','Circuit', 'Round']].copy()

df_heuristic1 = df_trunq.copy()

#get dummies for model predictions
dummies = pd.get_dummies(df_heuristic1['Circuit'],dtype = int)   # Create dummies
df_heuristic1 = pd.concat([df_heuristic1, dummies], axis=1).drop(['Circuit', 'Bahrain Grand Prix'], axis=1)  # Merge & drop
dummies = pd.get_dummies(df_heuristic1['TeamName'],dtype = int)   # Create dummies
df_heuristic1 = pd.concat([df_heuristic1, dummies], axis=1).drop(['TeamName', 'Mercedes'], axis=1)  # Merge & drop
dummies = pd.get_dummies(df_heuristic1['FullName'],dtype = int)   # Create dummies
df_heuristic1 = pd.concat([df_heuristic1, dummies], axis=1).drop(['FullName', 'Valtteri Bottas'], axis=1)  # Merge & drop

# Predict Top-10 when starting position is in the first 10 grid slots.
df_heuristic1['HeuristicPred'] = (df_heuristic1['GridPosition'] <= 10).astype(int)

#evaluate_baseline("=== Domain Heuristic Baseline (GridPosition <= 10) ===", df_heuristic1)

#### Why this feature should help predict our target

Normally, the top 10 finishers are a constant few, so considering this, they would aslo be the same ones to start the race inside the top10 grid positions. This can happen for a few reasons (higher resources, higher quality constructors, more experience, etc.).

However, this works best if data is only taken from the last few years, as huge changes can happen in Formula 1 across decades. Fortunately, we are predicting with recent data, so tthat should pose no issue for this feature.

### 2. "Driver rolling form (last 5 races) predicts Top-10"

In [7]:
# Baseline 2: Use each driver's Top-10 rate from the previous 5 races.

df_form = df_trunq[['FullName', 'Year', 'Round', 'Top10']].copy()
df_form = df_form.sort_values(['FullName', 'Year', 'Round']).reset_index(drop=True)

df_form['DriverTop10RateLast5'] = (
    df_form.groupby('FullName')['Top10']
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
)

# Global prior for first observed race of each driver.
global_top10_prior = df_train['Top10'].mean()
df_form['DriverTop10RateLast5'] = df_form['DriverTop10RateLast5'].fillna(global_top10_prior)

# New evaluate_baseline expects a dataframe with Top10, Year, and HeuristicPred.
df_baseline2 = df_form[['Top10', 'Year', 'DriverTop10RateLast5']].copy()
df_baseline2['HeuristicPred'] = (df_baseline2['DriverTop10RateLast5'] >= 0.5).astype(int)



#evaluate_baseline('Baseline 2: Driver rolling Top10 rate (last 5)', df_baseline2)

#### Why this feature should help predict our target

A driver's recent Top-10 rate captures current form, consistency, and adaptation to the season's car and rules. It's a clear display of skill and experience.

Drivers who have finished in the Top 10 repeatedly over their last few races are more likely to do so again than drivers with poor recent results. 

Using a rolling 5-race window also reduces noise from one-off outcomes (e.g., crashes, penalties, weather), making the signal more stable for predicting `Top10`.

### 3. "Constructor historical Top-10 rate predicts Top-10"

In [8]:
# Baseline 3: Use constructor-level historical Top-10 probability from training data.

train_mask = df_trunq['Year'].isin([2020,2021, 2022])
global_top10_prior = df_trunq.loc[train_mask, 'Top10'].mean()
team_top10_rate = df_trunq.loc[train_mask].groupby('TeamName')['Top10'].mean()

df_baseline3 = df_trunq[['Top10', 'Year', 'TeamName']].copy()
df_baseline3['TeamTop10Rate'] = df_baseline3['TeamName'].map(team_top10_rate).fillna(global_top10_prior)
df_baseline3['HeuristicPred'] = (df_baseline3['TeamTop10Rate'] >= 0.5).astype(int)

dummies = pd.get_dummies(df_baseline3['TeamName'],dtype = int)   # Create dummies
df_baseline3 = pd.concat([df_baseline3, dummies], axis=1).drop(['TeamName', 'Mercedes'], axis=1)  # Merge & drop


#evaluate_baseline('Baseline 3: Constructor Top10 historical rate', df_baseline3)

#### Why this feature should help predict our target

Constructor historical Top-10 rate reflects the underlying competitiveness and reliability of the team package (car performance, strategy quality, and pit execution). 

Since team-level performance strongly influences race outcomes, drivers in historically strong constructors are more likely to finish in the Top 10. 

This feature adds stable context beyond individual driver variation and helps model baseline race-day opportunity for `Top10`.

## Results/Comparissions

In [9]:
evaluate_baseline("Baseline 1: Start Top10, finish Top10", df_heuristic1)


Baseline 1: Start Top10, finish Top10
Training shape: (1220, 85)
Validation shape: (440, 85)
Testing shape: (479, 85)
              precision    recall  f1-score   support

  Not Top-10       0.74      0.59      0.66       239
      Top-10       0.66      0.80      0.72       240

    accuracy                           0.70       479
   macro avg       0.70      0.69      0.69       479
weighted avg       0.70      0.70      0.69       479



In [10]:
evaluate_baseline('Baseline 2: Driver rolling Top10 rate (last 5)', df_baseline2)
evaluate_baseline('Baseline 3: Constructor Top10 historical rate', df_baseline3)


Baseline 2: Driver rolling Top10 rate (last 5)
Training shape: (1220, 4)
Validation shape: (440, 4)
Testing shape: (479, 4)
              precision    recall  f1-score   support

  Not Top-10       0.77      0.78      0.77       239
      Top-10       0.78      0.76      0.77       240

    accuracy                           0.77       479
   macro avg       0.77      0.77      0.77       479
weighted avg       0.77      0.77      0.77       479


Baseline 3: Constructor Top10 historical rate
Training shape: (1220, 17)
Validation shape: (440, 17)
Testing shape: (479, 17)
              precision    recall  f1-score   support

  Not Top-10       0.71      0.42      0.53       239
      Top-10       0.59      0.82      0.69       240

    accuracy                           0.62       479
   macro avg       0.65      0.62      0.61       479
weighted avg       0.65      0.62      0.61       479



Now, we join all the features and check how they fare against eachother individually

In [11]:
df_model = df_trunq.copy()
#feature1
df_model['HeuristicPred1'] = (df_model['GridPosition'] <= 10).astype(int)
#feature2
df_form = df_form.sort_values(['FullName', 'Year', 'Round']).reset_index(drop=True)

df_form['DriverTop10RateLast5'] = (
    df_form.groupby('FullName')['Top10']
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
)

# Global prior for first observed race of each driver.
global_top10_prior = df_train['Top10'].mean()
df_model['DriverTop10RateLast5'] = df_form['DriverTop10RateLast5'].fillna(global_top10_prior)

df_model['HeuristicPred2'] = (df_model['DriverTop10RateLast5'] >= 0.5).astype(int)
# #feature3
df_model['TeamTop10Rate'] = df_model['TeamName'].map(team_top10_rate).fillna(global_top10_prior)
df_model['HeuristicPred3'] = (df_model['TeamTop10Rate'] >= 0.5).astype(int)


#get dummies
dummies = pd.get_dummies(df_model['Circuit'],dtype = int)   # Create dummies
df_model = pd.concat([df_model, dummies], axis=1).drop(['Circuit', 'Bahrain Grand Prix'], axis=1)  # Merge & drop
dummies = pd.get_dummies(df_model['TeamName'],dtype = int)   # Create dummies
df_model = pd.concat([df_model, dummies], axis=1).drop(['TeamName', 'Mercedes'], axis=1)  # Merge & drop
dummies = pd.get_dummies(df_model['FullName'],dtype = int)   # Create dummies
df_model = pd.concat([df_model, dummies], axis=1).drop(['FullName', 'Valtteri Bottas'], axis=1)  # Merge & drop



evaluate_baseline("ALL TOGETHER", df_model)


ALL TOGETHER
Training shape: (1220, 89)
Validation shape: (440, 89)
Testing shape: (479, 89)
              precision    recall  f1-score   support

  Not Top-10       0.73      0.71      0.72       239
      Top-10       0.72      0.74      0.73       240

    accuracy                           0.73       479
   macro avg       0.73      0.73      0.73       479
weighted avg       0.73      0.73      0.73       479



## Conclusions

This notebook builds and compares three practical heuristics for predicting whether a driver finishes in the Top 10: starting grid position, recent driver form (rolling last 5 races), and constructor historical Top-10 rate.

Across the analysis, the strongest signal comes from competitive context (grid position and team strength), while recent form adds useful temporal information that helps capture momentum across a season. Combining all features provides the most complete representation of race outcome likelihood, because it balances short-term performance with structural team-level advantage.

From a feature-engineering perspective, this confirms that Top-10 prediction benefits from mixing event-level, driver-level, and constructor-level signals rather than relying on a single heuristic. The resulting baseline is interpretable, reproducible, and a strong foundation for the next step: training more expressive models and validating whether they outperform these transparent heuristics on 2024 test races.